# Step 1: Prepare the GSDiff Model Environment
This cell sets up the required libraries, clones the original GSDiff repository, and downloads the required pretrained model weights (exactly as done in the original notebook).

In [ ]:
!pip install -q gdown networkx matplotlib pyvis Shapely Pillow

%cd /content
!git clone https://github.com/SizheHu/GSDiff.git
%cd GSDiff

!mkdir -p outputs
%cd outputs

!gdown "1pk7SmvLZ8ON3OUL3SNxPRu73ndVKru0z"
!gdown "1tExX8LdrFpJfBQH5y2emC6BltBwf9tHx"

!tar -xvf *.tar*
!unzip -q topo-params.zip
%cd /content

# Step 2: Download Your Custom Floorplan Generator Repository
Replace the placeholder `<YOUR_GITHUB_REPO_URL>` with your actual GitHub repository URL after you publish it. This will pull your Python structured engine.

In [ ]:
%cd /content
!git clone <YOUR_GITHUB_REPO_URL>

# CHANGE THIS to the name of your cloned repository folder!
REPO_FOLDER_NAME = "grad"
%cd {REPO_FOLDER_NAME}

# Install the door placement engine dependencies
!pip install -r door_placement/requirements.txt

# Step 3: Run the Pipeline
You can run your interactive python script right here in Colab. It will ask for your inputs directly underneath the cell!

In [ ]:
!python run_full_pipeline.py

# Alternative: Run Programmatically (Without CLI Prompt)
If you'd prefer to script it directly in Python without answering the interactive console prompts, you can do it like this:

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('.'))

from floorplan_generation.topology import generate_topology_from_form, visualize_agent_output
from floorplan_generation.inference import run_gsdiff_inference
from door_placement.pipeline import run_pipeline

user_input = {
    "property_type": "apartment",
    "rooms": {
        "living_rooms": 1,
        "bedrooms": 2,
        "master_bedrooms": 1,
        "bathrooms": 2,
        "kitchens": 1,
        "storage": 0
    }
}

# 1. Generate Topology
agent_nodes, agent_edges = generate_topology_from_form(user_input)
os.makedirs("outputs/custom_test", exist_ok=True)
visualize_agent_output(agent_nodes, agent_edges, save_path="outputs/custom_test/bubble_diagram.png")

# 2. Run GSDiff generation
json_dir = "outputs/custom_jsons"
run_gsdiff_inference(
    agent_nodes, agent_edges, 
    output_dir="outputs/custom_test", 
    json_dir=json_dir, 
    num_samples=15
)

# 3. Run Door Placement on specific plans (e.g. plan 0 and plan 5)
for plan_num in [0, 5]:
    input_json = os.path.join(json_dir, f"custom_pred_{plan_num}.json")
    out_path = f"outputs/final_door_placements/plan_{plan_num}"
    
    if os.path.exists(input_json):
        print(f"\nProcessing {input_json}...")
        run_pipeline(input_json=input_json, output_dir=out_path)
